file and import set up

In [2]:
import json
import fitz
import pandas as pd
import os
import spacy
print(os.getcwd())
main_path = "/Users/jaceysimpson/Vscode/FellowScript/"
bible_dict_path = os.path.join(main_path, "data/bible.json")
pdf_path = os.path.join(main_path, "data/ESV Bible.pdf")
book_list_path = os.path.join(main_path, "data/list_esv.csv")

doc = fitz.open(pdf_path)
nlp = spacy.load("en_core_web_sm")


/Users/jaceysimpson/Vscode/FellowScript/api/backend/bibleHandling


process PDF into lines

In [3]:
def process_pdf(out_file="data/list_esv.csv"):
    out_file = os.path.join(main_path, out_file)
    lines = []
    for page in doc:
        text = str(page.get_text())
        lines.extend(text.splitlines())
    df = pd.DataFrame({"lines": lines})
    df.to_csv(out_file)

process_pdf()

search book start and end

In [6]:
def get_start(start_idx, next_line, book_lines):
    idx = start_idx
    while "chapter" in next_line.lower() and idx < len(book_lines):
        idx += 1
        next_line = book_lines[idx]
    start = idx
    print(f"got start: {start}")
    return start

def get_end(start_idx, next_book, book_lines):
    idx = start_idx
    end = 0
    while next_book.lower() not in book_lines[idx].lower() and idx < len(book_lines):
        idx += 1
    end = idx
    print(f"got end: {end}")
    return end


search for book

In [12]:
def find_book(book: str, book_lines: pd.Series):
        idx = 0
        start, end = (0, 0)
        print(f"num lines in df: {len(book_lines)}")
        while idx < len(book_lines):
            line: str = book_lines.iloc[idx]
            if book.lower() in line.lower():
                if idx + 1 > len(book_lines) - 1:
                    idx += 1
                    continue
                next_line: str = book_lines[idx+1]
                start_idx = idx
                if "chapter" in next_line.lower() or (line.lower() == book.lower() and not next_line.isupper()):
                    start = get_start(start_idx, next_line, book_lines)
                    end = get_end(start, "footnotes", book_lines)
                    break
            idx += 1
                    
        return (start, end)

book = "obadiah"
book_lines = os.path.join(main_path, "data/list_esv.csv")
df = pd.read_csv(book_lines)
find_book(book, df["lines"])

num lines in df: 98407
got start: 73646
got end: 73747


(73646, 73747)

remove headers

In [14]:
def remove_stops(line):
    doc = nlp(line)
    filtered = [word.text for word in doc if not word.is_stop]
    return filtered

def is_header(line):
    words = remove_stops(line)
    is_header = True
    if len(words) < 2:
        return False
    for word in words:
        if not word[0].isupper() or word.isdigit():
            is_header = False
    return is_header

line = "The Creation of the World"
print(is_header(line))

True


parse the chapter

In [15]:
def new_chapter(line):
    if ":" in line:
        print(f"potential chapter: {line}")
        colon_idx = line.index(":")
        if colon_idx < len(line) - 1:
            if line[0].isdigit() and line[colon_idx+1].isdigit():
                print(f"line proves chapter".upper())
                return True
            else:
                print(f"line failed to prove as chapter".upper())
        else:
            print("line failed to prove as chapter".upper())
    return False

def parse_chapters(start: int, end: int, book_lines: pd.Series):
    idx = start
    chapters = []
    chapter = ""
    while idx < end:
        line = book_lines.loc[idx]
        if is_header(line):
            header = "HEAD::", line
            chapters.append(header)
            idx += 1
            continue
        if new_chapter(line) and len(chapter) > 0:
            chapters.append(chapter)
            chapter = line
        else:
            chapter += f" {line}"
        
        idx += 1
    chapters.append(chapter)
    return chapters
book = 'obadiah'
book_lines = os.path.join(main_path, "data/list_esv.csv")
df = pd.read_csv(book_lines)
start, end = find_book(book, df["lines"])
parse_chapters(start, end, df["lines"]) #type: ignore

num lines in df: 98407
got start: 73646
got end: 73747
potential chapter: Thus says the Lord GOD concerning Edom:
LINE FAILED TO PROVE AS CHAPTER
potential chapter: nations:
LINE FAILED TO PROVE AS CHAPTER


[('HEAD::', 'Edom Will Be Humbled'),
 ('HEAD::', "Edom's Violence Against Jacob"),
 ('HEAD::', 'The Day of the LORD Is Near'),
 ('HEAD::', 'The Kingdom of the LORD'),
 " OBADIAH 1 The vision of Obadiah. Thus says the Lord GOD concerning Edom: We have heard a report from the LORD,    and a messenger has been sent among the nations: “Rise up! Let us rise against her for battle!” 2Behold, I will make you small among the nations;    you shall be utterly despised.[1] 3The pride of your heart has deceived you,    you who live in the clefts of the rock,[2]    in your lofty dwelling, who say in your heart,    “Who will bring me down to the ground?” 4Though you soar aloft like the eagle,    though your nest is set among the stars,    from there I will bring you down,           declares the LORD. 5If thieves came to you,    if plunderers came by night—    how you have been destroyed!—    would they not steal only enough for themselves? If grape gatherers came to you,    would they not leave glea

pattern search for start of new chapter